# Chapter 6 Walkthrough: Functions, Scope, Decomposition & Modular Problem Solving

**Book Reference:** Chapter 6, Sections 6.1 – 6.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 6.3.1 — Anatomy of `def`, Parameters, `return`, and Docstrings

In [ ]:
def compute_bmi(weight_kg, height_cm):
    """Return Body Mass Index (kg/m^2) rounded to 2 decimal places."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    return round(bmi, 2)

# Reuse the same certified machine on multiple patients
bmi_a = compute_bmi(64.5, 168)
bmi_b = compute_bmi(54.6, 156)
print("Patient A BMI:", bmi_a, "| Patient B BMI:", bmi_b)
print("Docstring:", compute_bmi.__doc__)


## Section 6.3.2 — The #1 Beginner Trap: `return` vs. `print()`

In [ ]:
def show_bmi_only(weight_kg, height_cm):
    """Displays BMI on screen, but implicitly returns None!"""
    height_m = height_cm / 100
    print("Screen display:", round(weight_kg / (height_m ** 2), 2))

returned_good = compute_bmi(64.5, 168)
returned_bad = show_bmi_only(64.5, 168)

print("compute_bmi returned:", returned_good, type(returned_good))
print("show_bmi_only returned:", returned_bad, type(returned_bad))


## Section 6.3.3 — Default Parameters & Keyword Arguments

In [ ]:
def classify_anemia(hb_g_dl, pregnant=True):
    """Classify maternal or adult anemia using WHO hemoglobin cutoffs."""
    cutoff = 11.0 if pregnant else 12.0
    if hb_g_dl < cutoff:
        return "Anemic"
    return "Normal"

print("Pregnant (default), 10.6 ->", classify_anemia(10.6))
print("Pregnant (default), 11.4 ->", classify_anemia(11.4))
print("Non-pregnant, 11.4       ->", classify_anemia(11.4, pregnant=False))


## Section 6.3.4 — Local vs. Global Scope & Call-Frame Isolation

In [ ]:
FEVER_CUTOFF_C = 38.0   # Read-only global constant (UPPER_CASE)
patient_status = "Waiting"

def screen_temperature(temp_c):
    """Return fever status using local scope without mutating globals."""
    patient_status = "Febrile" if temp_c >= FEVER_CUTOFF_C else "Normal"
    return patient_status

lab_result = screen_temperature(38.6)
print("Returned from function:", lab_result)
print("Outer global unchanged:", patient_status)


## Section 6.3.5 — Decomposition: Composing Small Functions Together

In [ ]:
def is_febrile(temp_c):
    """Return True if temperature meets fever threshold (>= 38.0 C)."""
    return temp_c >= 38.0

def assess_mother(patient_id, temp_c, hb_g_dl):
    """Compose smaller helpers into a single clear triage summary."""
    fever_flag = is_febrile(temp_c)
    anemia_label = classify_anemia(hb_g_dl, pregnant=True)
    needs_review = fever_flag or (anemia_label == "Anemic")
    return (patient_id, anemia_label, fever_flag, needs_review)

print(assess_mother("UHC-0042", 38.4, 10.2))
print(assess_mother("UHC-0043", 36.9, 12.1))


## Section 6.3.6 — Break It On Purpose! (Function & Scope Error Anatomy)

In [ ]:
# Intentional Bug #1: TypeError — missing a required positional argument
bmi_incomplete = compute_bmi(64.5)


In [ ]:
# Intentional Bug #2: NameError — reading a local variable outside its def
def calc_daily_dose(weight_kg):
    daily_mg = weight_kg * 15
    return daily_mg

dose = calc_daily_dose(20.0)
print(daily_mg)


In [ ]:
# Intentional Bug #3: TypeError — math on None from a print()-only function
def print_only_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    print(round(weight_kg / (height_m ** 2), 2))

bad_bmi = print_only_bmi(64.5, 168)
bmi_gap = bad_bmi - 25.0


## Section 6.4 — Worked Problem-Structuring Blueprint: Modular Outbreak Surveillance Pipeline

In [ ]:
def clean_daily_cases(raw_counts):
    """Return list of valid case counts, skipping None and 999 sentinels."""
    valid = []
    for count in raw_counts:
        if count is None or count == 999:
            continue
        valid.append(count)
    return valid

def safe_incidence_rate(total_cases, population, per=100000):
    """Return incidence rate per N people, or None if population <= 0."""
    if population <= 0:
        return None
    return round((total_cases / population) * per, 2)

def summarize_district(district, raw_counts, population):
    """Compose cleaning + rate helpers into a district summary tuple."""
    valid = clean_daily_cases(raw_counts)
    total = sum(valid)
    rate = safe_incidence_rate(total, population)
    alert = (rate is not None) and (rate >= 35.0)
    return (district, len(valid), total, rate, alert)

rep_dhaka = summarize_district("Dhaka", [18, None, 27, 999, 34, 21], 250000)
rep_empty = summarize_district("Unknown", [10, 12], 0)
print("Dhaka report:  ", rep_dhaka)
print("Zero-pop guard:", rep_empty)
